In [1]:
from pathlib import Path
import sys

# Find the project root from the notebook's working directory.
PROJECT_ROOT = Path.cwd().resolve()

# Run until found the (PROJECT_ROOT / "requirements.txt")
while not (PROJECT_ROOT / "requirements.txt").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Cannot find the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow imports starting with "src".
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import torch
import joblib
import numpy as np
from src.models.mlp import MLP

MODELS_DIR = PROJECT_ROOT / 'models'

def load_mlp_model(filename="mlp_model.pth"):
    path = MODELS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
        
    checkpoint = torch.load(path, map_location=torch.device('cpu'))
    config = checkpoint['config']
    
    # Khởi tạo lại kiến trúc mạng y hệt lúc train
    model = MLP(
        sizes=config['sizes'],
        activation=config['activation'],
        p_drop=config['p_drop'],
        init=config['init'],
        seed=config['seed']
    )
    
    model.model.load_state_dict(checkpoint['model_state_dict'])
    
    model.model.eval()
    print(f"Đã tải MLP từ {filename} (Epoch tốt nhất: {checkpoint.get('best_epoch')})")
    return model

def load_sklearn_model(filename):
    path = MODELS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    model = joblib.load(path)
    print(f"Đã tải {filename}")
    return model

def load_ensemble_model(dirname="ensemble_v1"):
    ens_dir = MODELS_DIR / dirname
    if not ens_dir.exists():
        raise FileNotFoundError(f"Không tìm thấy thư mục: {ens_dir}")
    
    import json
    with open(ens_dir / "config.json", "r") as f:
        config = json.load(f)
    
    ensemble = CustomSoftVotingEnsemble(weights=config['weights'])
    
    mlp_checkpoint = torch.load(ens_dir / "mlp_part.pth", map_location='cpu')
    ensemble.mlp.model.load_state_dict(mlp_checkpoint['model_state_dict'])
    ensemble.mlp.model.eval()
    
    ensemble.rf = joblib.load(ens_dir / "rf_part.joblib")
    ensemble.knn = joblib.load(ens_dir / "knn_part.joblib")
    
    print(f"Đã tải Ensemble từ {dirname}/ (Weights: {config['weights']})")
    return ensemble

In [6]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"
import numpy as np
import time
from src.preprocessing.scaler import split_val, scale_flat 

train_data = np.load(DATA_DIR / "mnist_train.npz")
test_data = np.load(DATA_DIR / "mnist_test.npz")

X_train, y_train = train_data["X"], train_data["y"]
X_test, y_test = test_data["X"], test_data["y"]

X_train, X_val, y_train, y_val = split_val(X=X_train,y=y_train, val_size=0.1, seed=43)
X_train, X_val, X_test,tools = scale_flat(X_train=X_train, X_test=X_test, X_val=X_val, mode='raw')
print(f'Train: {len(X_train)}, Test: {len(X_test)}, Val: {len(X_val)}')

Train: 54000, Test: 10000, Val: 6000


In [7]:
model_mlp = load_mlp_model()

Đã tải MLP từ mlp_model.pth (Epoch tốt nhất: 9)


In [8]:
y_pred = model_mlp.predict(X_test)

In [9]:
from src.evaluation.metrics import ClassificationMetrics

In [10]:
metrics = ClassificationMetrics(y_pred, y_test)

In [13]:
metrics.report()

     Class  Precision     Recall         F1
         0      0.991      0.983      0.987 
         1      0.989      0.991      0.990 
         2      0.983      0.979      0.981 
         3      0.979      0.978      0.979 
         4      0.987      0.987      0.987 
         5      0.983      0.979      0.981 
         6      0.985      0.981      0.983 
         7      0.975      0.984      0.979 
         8      0.975      0.973      0.974 
         9      0.971      0.982      0.977 
 Macro avg      0.982      0.982      0.982 

Accuracy: 0.982
